<a href="https://colab.research.google.com/github/HarishSivakumaran/ML-Projects/blob/main/ml_algos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Per-Sample Gradients for DP-SGD: Given a batch of $B=4$ 1D inputs $x$ and targets $y$, write a model function $f(w, x) = w \cdot x$ with loss $\mathcal{L}_i = (f(w, x_i) - y_i)^2$. Use jax.vmap(jax.grad(...)) to calculate 4 individual gradient scalars, clip each gradient norm to $1.0$, and average them.

In [16]:
import jax
import jax.numpy as jnp

# let X be CxB, column vectors
seed = jax.random.key(42)

k1, k2, k3 = jax.random.split(seed, 3)
DIM = 7
B= 4
x = jax.random.normal(k1, (DIM, B))
y = jax.random.normal(k1, (1, B))

params = {
    'w': jax.random.normal(k2, (1,DIM)),
    'b': jnp.array([1.0]),
}

def loss(params, x, y):
  return jnp.square((jnp.dot(params['w'], x) + params['b']) - y).squeeze()

parallel_single_grad_loss = jax.vmap(jax.grad(loss), in_axes=(None, 1, 1))

def step(params, x, y):
  print(f'input: {x}')
  print(f'label: {y}')
  print(f' params: {params}')
  grads_per_sample = parallel_single_grad_loss(params, x, y)
  print(grads_per_sample)
  w_grads = grads_per_sample['w'].squeeze()
  b_grads = grads_per_sample['b'].squeeze()
  print(f'w_grads: {w_grads}')
  print(f'b_grads: {b_grads}')
  w_grads_norm = jnp.linalg.norm(w_grads, axis=(0))
  b_grads_norm = jnp.linalg.norm(b_grads, axis=(0))
  print(f'w_grads_norm: {w_grads_norm}')
  print(f'b_grads_norm: {b_grads_norm}')
  clip_norm = 1.0
  clipped_grads = w_grads / jnp.maximum(1.0, w_grads_norm / clip_norm)
  print(f'clipped_grads: {clipped_grads}')
  print(f'normalized grads norm: {jnp.linalg.norm(clipped_grads, axis=(0))}')
  avg_grads = jnp.mean(clipped_grads, axis=(0))
  print(f'avg_grads: {avg_grads}')



step(params, x, y)



input: [[ 0.07592554 -0.48634264  1.2903206   0.5196119 ]
 [ 0.30040437  0.31034866  0.5761609  -0.8074621 ]
 [-1.9883217   0.6395295   0.21763174  0.00247425]
 [ 1.6645706   0.20313536 -0.02138225 -0.68679047]
 [ 0.01922452  0.47158983 -0.84080535  1.0207201 ]
 [-2.1002991   0.43098864 -0.69074976  1.0829577 ]
 [ 0.19644964  0.85078067  1.8178146   0.71369797]]
label: [[ 0.07592554 -0.48634264  1.2903206   0.5196119 ]]
 params: {'w': Array([[ 0.60576403,  0.7990441 , -0.908927  , -0.63525754, -1.2226585 ,
        -0.83226097, -0.47417238]], dtype=float32), 'b': Array([1.], dtype=float32)}
{'b': Array([[ 7.182506],
       [-1.218636],
       [ 3.016807],
       [-3.807438]], dtype=float32), 'w': Array([[[ 5.4533565e-01,  2.1576562e+00, -1.4281133e+01,  1.1955789e+01,
          1.3808022e-01, -1.5085411e+01,  1.4110007e+00]],

       [[ 5.9267467e-01, -3.7820205e-01, -7.7935374e-01, -2.4754806e-01,
         -5.7469636e-01, -5.2521831e-01, -1.0367919e+00]],

       [[ 3.8926485e+00,  1.7

💻 Day 3 Hands-On Practice Exercisesvalue_and_grad Training Step: Write a function step(params, x, y, lr) that takes a parameter PyTree {'w': w, 'b': b}, computes loss_val, grads using jax.value_and_grad, applies an SGD update ($p \leftarrow p - \text{lr} \cdot g$) via jax.tree.map, and returns (updated_params, loss_val).

In [5]:
import jax
import jax.numpy as jnp

def loss(params, x, y):
  return jnp.mean(jnp.square((jnp.dot(x, params['w']) + params['b']) - y))

def step(params, x, y, lr):
  loss_val, grad = jax.value_and_grad(loss)(params, x, y)
  updated_params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grad)
  return updated_params, loss_val

k1, k2, k3 = jax.random.split(jax.random.key(42), 3)
x = jax.random.normal(k1, (10, 15))
y = jax.random.normal(k2, (10, ))
params = {
    'w': jax.random.normal(k3, (15,1)),
    'b': jnp.array([1.0]),
}


print(step(params, x, y, 10e-3))


({'b': Array([0.9579859], dtype=float32), 'w': Array([[ 0.3644766 ],
       [ 0.5649586 ],
       [-1.06929   ],
       [-0.33437946],
       [-0.17365052],
       [-1.6051906 ],
       [-1.7569315 ],
       [-0.43997833],
       [-0.07641385],
       [-1.7343235 ],
       [ 1.385112  ],
       [ 0.17762288],
       [ 0.96262264],
       [-0.20404385],
       [-0.8912208 ]], dtype=float32)}, Array(37.603073, dtype=float32))


💻 Day 2 Hands-On Practice ExercisesTracing vs. Runtime Print Challenge: Write a @jax.jit function that computes $y = x^2 + 1$. Add both a standard print(x) statement and a jax.debug.print("Runtime x: {}", x) statement. Run it twice with different inputs and explain what is output on each call910.Fixing Control Flow: Take the following function that throws an error when wrapped in @jax.jit, and fix it using static_argnames:def scale_mode(x, mode="double"):
    if mode == "double":
        return x * 2.0
    else:
        return x * 0.5

In [5]:
import jax
import jax.numpy as jnp

def apply(x):
  print(x)
  jax.debug.print("Runtime x: {}", x)
  return jnp.square(x) + 1

print(apply(jnp.array([2.0, 3.0, 4.0])))
print(apply(5.0))
print(apply(9))

jit_apply = jax.jit(apply)

print(jit_apply(jnp.array([2.0, 3.0, 4.0])))
print(jit_apply(jnp.array([3.0, 4.0, 5.0])))
print(jit_apply(5.0))
print(jit_apply(6.0))
print(jit_apply(9))
print(jit_apply(17))


[2. 3. 4.]
Runtime x: [2. 3. 4.]
[ 5. 10. 17.]
5.0
Runtime x: 5.0
26.0
9
Runtime x: 9
82
JitTracer(float32[3])
Runtime x: [2. 3. 4.]
[ 5. 10. 17.]
Runtime x: [3. 4. 5.]
[10. 17. 26.]
JitTracer(~float32[])
Runtime x: 5.0
26.0
Runtime x: 6.0
37.0
JitTracer(~int32[])
Runtime x: 9
82
Runtime x: 17
290


In [16]:
import jax
import jax.numpy as jnp

@jax.jit(static_argnames=("mode"))
def scale_mode(x, mode="double"):
  print(x)
  jax.debug.print("Runtime x: {}", x)
  if mode == "double":
      return x * 2.0
  else:
      return x * 0.5


scale_mode(jnp.array([2.0, 3.0, 4.0]), mode="boo")
scale_mode(jnp.array([1.0, 3.0, 4.0]), mode="boo")

JitTracer(float32[3])
Runtime x: [2. 3. 4.]
Runtime x: [1. 3. 4.]


Array([0.5, 1.5, 2. ], dtype=float32)

In [25]:
import jax
import jax.numpy as jnp

seed = jax.random.key(42)

gaussian = jax.random.normal(seed, (2, 3)) * 100
print(gaussian)

def square_sample_naive(x):
  for i in range(x.shape[0]):
    x = x.at[i].set(jnp.square(x[i]))
  return x

print(square_sample_naive(gaussian))

def square_sample(x):
  return jnp.square(x)

vmap_square = jax.vmap(square_sample)
print(vmap_square(gaussian))


print(jax.make_jaxpr(square_sample_naive)(gaussian))
print(jax.make_jaxpr(vmap_square)(gaussian))


[[ -2.8304615  46.713184   29.570297 ]
 [ 15.354591  -12.403282   21.692314 ]]
[[   8.011513 2182.1216    874.40247 ]
 [ 235.76347   153.84142   470.5565  ]]
[[   8.011513 2182.1216    874.40247 ]
 [ 235.76347   153.84142   470.5565  ]]
{ lambda ; a:f32[2,3]. let
    b:f32[1,3] = slice[limit_indices=(1, 3) start_indices=(0, 0) strides=None] a
    c:f32[3] = squeeze[dimensions=(0,)] b
    d:f32[3] = square c
    e:i32[1] = broadcast_in_dim 0:i32[]
    f:f32[2,3] = scatter[
      dimension_numbers=ScatterDimensionNumbers(update_window_dims=(0,), inserted_window_dims=(0,), scatter_dims_to_operand_dims=(0,), operand_batching_dims=(), scatter_indices_batching_dims=())
      indices_are_sorted=True
      mode=GatherScatterMode.FILL_OR_DROP
      unique_indices=True
      update_consts=()
      update_jaxpr=None
    ] a e d
    g:f32[1,3] = slice[limit_indices=(2, 3) start_indices=(1, 0) strides=None] f
    h:f32[3] = squeeze[dimensions=(0,)] g
    i:f32[3] = square h
    j:i32[1] = broadcast

💻 Day 1 Hands-On Practice ExercisesArray Mutation Challenge: Write a function that takes a 1D array of length $N$ and replaces all negative values with $0.0$ using .at[].set() and jnp.where.PRNG Pipeline: Build a key-splitting loop that generates $5$ distinct mini-batches of Gaussian noise without ever reusing a key.PyTree Optimizer: Define a nested dictionary with weights, biases, and hyperparameters. Write a custom jax.tree.map call that computes the L2 norm ($\sum w^2$) of all weight arrays while ignoring scalar learning rates.

In [ ]:
import jax
import jax.numpy as jnp

def relu(arr: jnp.array):
  return jnp.where(arr >= 0, arr, 0.0)

test = jnp.array([-1.0, 0, 1.0, 2, 3])

print(relu(test))
in_place = test.at[jnp.where(test < 0)].set(0.0)
print(jnp.where(test >= 0))
print(in_place)
print(test)

[0. 0. 1. 2. 3.]
(Array([1, 2, 3, 4], dtype=int32),)
[0. 0. 1. 2. 3.]
[-1.  0.  1.  2.  3.]


In [ ]:
import jax
import jax.numpy as jnp


def gaussian_sample(key, shape):
  return jax.random.normal(key, shape)

root_key = jax.random.key(42)

for i in range(5):
  k1, k2 = jax.random.split(root_key, 2)
  print(gaussian_sample(k1, (10, )))
  root_key = k2

[ 0.07592554 -0.48634264  1.2903206   0.5196119   0.30040437  0.31034866
  0.5761609  -0.8074621  -1.9883217   0.6395295 ]
[-0.74412036  1.52171     0.1847949  -1.1812532  -0.67319936 -0.32261458
  0.05882881  1.5566975   2.0083923   0.97201616]
[-0.92217225  1.2502675  -1.4594661  -0.6433759   0.10886536  0.5296943
 -1.7170644   0.30685094  0.5052932  -0.4425835 ]
[-0.8789221   1.7597361  -1.7431877   0.28519595  0.13290869 -2.3614807
  0.99604434 -0.21629068  3.365236   -0.03634243]
[-0.24331857 -0.05206627 -0.5459883   0.5970541   0.3256138  -0.21120721
 -0.5110182   1.2065462  -1.715144   -0.6556372 ]


In [ ]:
import jax
import jax.numpy as jnp

params = {
    "weights": jnp.array([[1,2,3], [4,5,6]]),
    "biases": jnp.array([0.1, 0.2, 0.3]),
    "learning_rate": 0.01
}

def calc_l2(params):
  def leaf_with_path(path, leaf):
    keys = [p.key for p in path]
    if 'weights' in keys and isinstance(leaf, jax.Array):
      return jnp.sum(jnp.square(leaf))
    else:
      return jnp.array(0.0)
  return jax.tree_util.tree_map_with_path(leaf_with_path, params)

print(calc_l2(params))



{'biases': Array(0., dtype=float32, weak_type=True), 'learning_rate': Array(0., dtype=float32, weak_type=True), 'weights': Array(91, dtype=int32)}
